In [6]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime
# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

#데이터 불러오기 
df = pd.read_csv('./model_df_new_cat_음수할인율처리후.csv')
print(df.columns)
df.head(1)


Index(['기획년도', '주차', '카테고리_통합', '시즌이월', '시즌', '총입고수량', '판매수량', '판매액', '평균택가',
       '평균원가', '주차별_평균_실판매가', '월별_평균_실판매가', '시즌별_평균_실판매가', '총입고원가', '총입고택가',
       '매출원가계', '판매택가계', '실판매가', '할인율', '누적판매수량', '누적판매액', '누적매출원가', '누적판매택가',
       '누적판매율', 'ROI', '악천후일수', '평균기온(도)'],
      dtype='object')


,기획년도,주차,카테고리_통합,시즌이월,시즌,총입고수량,판매수량,판매액,평균택가,평균원가,...,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,악천후일수,평균기온(도)
0,2021,2021-07-11,가을_가성비 수트셋업,01_시즌,가을,13163,496,84636400,184000,28105,...,170638,7,496,84636400,13940080,91264000,3.77,0.17,0,28.5


In [7]:
# 피벗테이블로 4년동안 데이터가 있는 카테고리만 남김
# 카테고리별 연도 존재 여부 확인 (0: 없음, 1: 있음)
category_year_table = df.groupby(["카테고리_통합", "기획년도"]).size().unstack(fill_value=0)
# 연도가 존재하면 1로 변환 (카테고리가 존재했음을 의미)
category_year_table = (category_year_table > 0).astype(int)

# 21, 22, 23, 24년도 모두 존재한 카테고리만 필터링
categories_all_years = category_year_table[
    (category_year_table.get(2021, 0) == 1) & 
    (category_year_table.get(2022, 0) == 1) & 
    (category_year_table.get(2023, 0) == 1) & 
    (category_year_table.get(2024, 0) == 1)
].index

# 새로운 데이터프레임 생성
df = df[df["카테고리_통합"].isin(categories_all_years)].copy()

# 2023년도 데이터(지난 1년동안의 데이터)로 카테고리 선정하고자 함
# 판매 중간에 연도가 바뀌면서 잘린 겨울 제품 제외함함
df = df[(df['기획년도'] == 2023) & (df['시즌'] != '겨울')]

In [8]:
# 기준 1) ----------------------------------------------------------------------------------------------------------------
# 매출 기여도가 높은 핵심 카테고리 (판매량 & 매출 기준)
top_sales = df.groupby("카테고리_통합", group_keys=False).agg(
    총판매수량=("판매수량", "sum"),
    총매출=("판매액", "sum"),
    총입고수량=("총입고수량", "max")
).reset_index()

# 기준 2) ----------------------------------------------------------------------------------------------------------------
# 재고 부담 & 할인 전략 개선이 필요한 카테고리(최대 누적판매율 & 평균 할인율)
top_inventory_adjusted = df.groupby("카테고리_통합", group_keys=False).agg(
    최대누적판매율=("누적판매율", "max"),  
    평균할인율=("할인율", "mean")  
).reset_index()

# 시즌 종료 시 할인율 상승 패턴 확인
df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 각 카테고리별 주차별 할인율 변화 측정
discount_trend = df.groupby(["카테고리_통합", "연도주차"], group_keys=False)["할인율"].mean().reset_index()
discount_trend["할인율_변화"] = discount_trend.groupby("카테고리_통합", group_keys=False)["할인율"].diff()  

# 마지막 4주 동안 할인율이 과하게 상승한 카테고리 찾기
last_weeks = discount_trend.groupby("카테고리_통합", group_keys=False).tail(4)  
discount_rise = last_weeks.groupby("카테고리_통합", group_keys=False)["할인율_변화"].sum().reset_index()
discount_rise.columns = ["카테고리_통합", "시즌 종료 할인율 증가량"]

# 마지막 주차의 할인율 추가
last_week_discount = df.groupby("카테고리_통합", group_keys=False).apply(
    lambda x: x.loc[x["주차"] == x["주차"].max(), "할인율"].mean()
).reset_index()
last_week_discount.columns = ["카테고리_통합", "마지막 주차 할인율"]

# 기준 3) ----------------------------------------------------------------------------------------------------------------
# 가격 탄력성이 높은 카테고리 (할인율-판매량 상관관계)
top_correlation = df.groupby("카테고리_통합", group_keys=False).apply(
    lambda x: x["할인율"].corr(x["판매수량"])
).reset_index()
top_correlation.columns = ["카테고리_통합", "가격 탄력성(할인율-판매량 상관관계)"]

# 가격 변화율과 판매량 변화율을 이용한 가격 탄력성 계산
df["가격 변화율"] = df.groupby('카테고리_통합')['주차별_평균_실판매가'].pct_change() * 100
df["판매량 변화율"] = df.groupby('카테고리_통합')['판매수량'].pct_change() * 100

df["가격 탄력성(판매량변화율/가격변화율)"] = df["판매량 변화율"] / df["가격 변화율"]
df["가격 탄력성(판매량변화율/가격변화율)"] = df["가격 탄력성(판매량변화율/가격변화율)"].replace([np.inf, -np.inf], np.nan)

elastic_cate = df.groupby('카테고리_통합')['가격 탄력성(판매량변화율/가격변화율)'].mean().reset_index()

# 모든 데이터 병합 ----------------------------------------------------------------------------------------------------------------
category_selection = (
    top_sales
    .merge(top_inventory_adjusted, on="카테고리_통합", how="left")
    .merge(last_week_discount, on="카테고리_통합", how="left")
    .merge(top_correlation, on="카테고리_통합", how="left")
    .merge(discount_rise, on="카테고리_통합", how="left")
    .merge(elastic_cate, on="카테고리_통합", how="left")
    .fillna(0)  
)

# 소수점 라운딩
category_selection["평균할인율"] = category_selection["평균할인율"].round(2)
category_selection["가격 탄력성(할인율-판매량 상관관계)"] = category_selection["가격 탄력성(할인율-판매량 상관관계)"].round(2)
category_selection["가격 탄력성(판매량변화율/가격변화율)"] = category_selection["가격 탄력성(판매량변화율/가격변화율)"].round(2)
category_selection["마지막 주차 할인율"] = category_selection["마지막 주차 할인율"].astype(int)

# 정렬 및 순위 계산 (최대 누적판매율은 낮을수록 우선순위 → 오름차순, 나머지는 높을수록 우선순위 → 내림차순)
category_selection["총매출_순위"] = category_selection["총매출"].rank(method="min", ascending=False).astype(int)
category_selection["총입고_순위"] = category_selection["총입고수량"].rank(method="min", ascending=False).astype(int)
category_selection["최대누적판매율_순위"] = category_selection["최대누적판매율"].rank(method="min", ascending=True).astype(int)  # 낮을수록 재고 부담 ↑
category_selection["평균할인율_순위"] = category_selection["평균할인율"].rank(method="min", ascending=False).astype(int)
category_selection["시즌 종료 할인율 증가량_순위"] = category_selection["시즌 종료 할인율 증가량"].rank(method="min", ascending=False).astype(int)
category_selection["가격탄력성(상관계수) 순위"] = category_selection["가격 탄력성(할인율-판매량 상관관계)"].rank(method="min", ascending=False).astype(int) 
category_selection["가격탄력성(판매량변화율) 순위"] = category_selection["가격 탄력성(판매량변화율/가격변화율)"].rank(method="min", ascending=True).astype(int) # 낮을수록 탄력성 좋음음

# 각 기준별 점수 계산 ----------------------------------------------------------------------------------------------------------------
# 기준 1: 매출 기여도 높은 카테고리 (총매출, 총입고수량)
category_selection["기준1_점수"] = category_selection["총매출_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["총입고_순위"].rank(method="min", ascending=True).astype(int)

# 기준 2: 재고 부담 & 할인 전략 개선 (최대누적판매율, 평균할인율, 시즌 종료 할인율 증가량)
category_selection["기준2_점수"] = category_selection["최대누적판매율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["평균할인율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["시즌 종료 할인율 증가량_순위"].rank(method="min", ascending=True).astype(int)

# 기준 3: 가격 탄력성 2가지 합산
category_selection["기준3_점수"] = category_selection["가격탄력성(상관계수) 순위"].rank(method="min", ascending=True).astype(int) + \
                                 category_selection["가격탄력성(판매량변화율) 순위"].rank(method="min", ascending=True).astype(int)

# 기준별 순위 계산 (오름차순, 낮을수록 우선순위)
category_selection["기준1_순위"] = category_selection["기준1_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준2_순위"] = category_selection["기준2_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준3_순위"] = category_selection["기준3_점수"].rank(method="min", ascending=True).astype(int)

# 최종 순위 계산 (기준 1, 2, 3 순위를 모두 합산)
category_selection["최종_순위"] = category_selection["기준1_순위"] + category_selection["기준2_순위"] + category_selection["기준3_순위"]
category_selection["최종_순위"] = category_selection["최종_순위"].rank(method="min", ascending=True).astype(int)

# 최종 출력: 순위만 포함 ----------------------------------------------------------------------------------------------------------------
final_rank_output = category_selection[[
    "카테고리_통합","기준1_순위", "기준2_순위", "기준3_순위", "최종_순위","총판매수량","총매출","총입고수량","최대누적판매율","평균할인율","마지막 주차 할인율",
    "시즌 종료 할인율 증가량","가격 탄력성(할인율-판매량 상관관계)","가격 탄력성(판매량변화율/가격변화율)","총매출_순위","총입고_순위","최대누적판매율_순위",
    "평균할인율_순위","시즌 종료 할인율 증가량_순위","가격탄력성(상관계수) 순위","가격탄력성(판매량변화율) 순위"]].sort_values(by="최종_순위")

# 기준 점수를 기준으로 정렬, 하나씩 주석 해제하면서 정렬
# final_rank_output.sort_values(by=["기준1_순위"])
# final_rank_output.sort_values(by=["기준2_순위"])
# final_rank_output.sort_values(by=["기준3_순위"])
final_rank_output #최종순위대로 정렬

,카테고리_통합,기준1_순위,기준2_순위,기준3_순위,최종_순위,총판매수량,총매출,총입고수량,최대누적판매율,평균할인율,...,시즌 종료 할인율 증가량,가격 탄력성(할인율-판매량 상관관계),가격 탄력성(판매량변화율/가격변화율),총매출_순위,총입고_순위,최대누적판매율_순위,평균할인율_순위,시즌 종료 할인율 증가량_순위,가격탄력성(상관계수) 순위,가격탄력성(판매량변화율) 순위
20,여름_캐주얼셔츠,2,5,2,1,97697,2959959897,158857,61.50,62.00,...,-2.0,0.67,-0.04,5,1,12,4,10,3,9
21,여름_팬츠,7,11,1,2,78720,2886590147,118291,66.55,57.31,...,-1.0,0.67,-5.15,9,5,16,10,9,3,6
17,여름_반팔티셔츠,5,8,7,3,107211,2923225342,137229,78.13,64.93,...,0.0,0.54,-2.35,7,3,21,2,8,8,7
18,여름_자켓,7,14,5,4,20913,3129651456,30109,69.46,47.76,...,1.0,0.20,-1038.68,3,11,19,13,7,13,1
11,사계절_셋업셔츠,3,8,15,4,87208,2948580508,147470,59.14,51.00,...,-2.0,0.27,10.79,6,2,10,11,10,12,14
9,사계절_가성비 수트셋업,1,21,4,4,76589,8830897666,120246,63.69,39.36,...,-7.0,0.74,5.08,1,4,13,18,19,1,12
4,봄_가디건,19,2,8,7,6606,415030305,13201,50.04,60.00,...,3.0,0.65,3.54,19,19,5,6,5,5,11
0,가을_가성비 수트셋업,9,14,8,8,21115,2922341852,31018,68.07,34.50,...,9.0,0.20,-25.66,8,10,18,20,1,13,3
3,가을_팬츠,17,4,10,8,10475,543351816,22020,47.57,41.86,...,7.0,0.64,3.03,17,16,3,16,3,7,10
16,여름_반팔스웨터,5,18,11,10,77779,3003857821,93167,83.48,58.00,...,-4.0,0.45,-2.16,4,6,22,8,15,11,8


In [ ]:
# final_rank_output.to_csv('final_rank_output.csv', index=False, encoding='utf-8-sig')